<a href="https://colab.research.google.com/github/kangna31/greeting/blob/main/Shopping_Mall_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: IMPORT LIBRARIES AND CREATE DATASETS

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime
import uuid
import os
import matplotlib.pyplot as plt

# Starting balance
INITIAL_BALANCE = 3000
balance = INITIAL_BALANCE

# Original shopping mall products
products_data = [
    [1, "Orange Juice", "Beverages", 70, 200, "🍊", "#ff6b6b"],
    [2, "Apple Juice", "Beverages", 80, 150, "🍎", "#ffd93d"],
    [3, "Cold Coffee", "Beverages", 60, 180, "☕", "#6bcb77"],
    [4, "Mineral Water", "Beverages", 20, 300, "💧", "#4d96ff"],
    [5, "Apple", "Fruits", 30, 250, "🍎", "#ff6b6b"],
    [6, "Banana", "Fruits", 10, 400, "🍌", "#ffd93d"],
    [7, "Orange", "Fruits", 20, 200, "🍊", "#ff6b6b"],
    [8, "Grapes", "Fruits", 50, 150, "🍇", "#9b59b6"],
    [9, "Potato", "Vegetables", 20, 300, "🥔", "#e67e22"],
    [10, "Tomato", "Vegetables", 30, 250, "🍅", "#e74c3c"],
    [11, "Carrot", "Vegetables", 25, 200, "🥕", "#ff6b6b"],
    [12, "Broccoli", "Vegetables", 40, 150, "🥦", "#6bcb77"],
    [13, "Milk", "Dairy Products", 30, 200, "🥛", "#4d96ff"],
    [14, "Butter", "Dairy Products", 55, 100, "🧈", "#ffd93d"],
    [15, "Cheese", "Dairy Products", 80, 80, "🧀", "#ffd93d"],
    [16, "Yogurt", "Dairy Products", 20, 150, "🥣", "#ff6b6b"],
    [17, "Notebook", "Stationery", 50, 200, "📓", "#4d96ff"],
    [18, "Pen", "Stationery", 10, 500, "🖊️", "#3498db"],
    [19, "Pencil", "Stationery", 5, 600, "✏️", "#ffd93d"],
    [20, "Eraser", "Stationery", 5, 400, "🧽", "#ff6b6b"]
]

# Create products.csv only if it does not already exist
if not os.path.exists("products.csv"):
    products_df = pd.DataFrame(
        products_data,
        columns=["Product_ID", "Name", "Category", "Price",
                 "Stock", "Emoji", "Color"]
    )
    products_df.to_csv("products.csv", index=False)

# Create customers.csv
if not os.path.exists("customers.csv"):
    customers_df = pd.DataFrame(columns=[
        "Customer_ID", "Name", "Phone",
        "Total_Spent", "Visits", "Last_Visit"
    ])
    customers_df.to_csv("customers.csv", index=False)

# Create sales.csv
if not os.path.exists("sales.csv"):
    sales_df = pd.DataFrame(columns=[
        "Transaction_ID", "Date", "Customer_Name", "Phone",
        "Product_Name", "Quantity", "Unit_Price", "Total"
    ])
    sales_df.to_csv("sales.csv", index=False)

# Load product dataset
products_df = pd.read_csv("products.csv")

# Shopping cart
cart = {}

print("✅ Shopping Mall datasets created successfully!")
print("\nAvailable Products:")
display(products_df)

✅ Shopping Mall datasets created successfully!

Available Products:


,Product_ID,Name,Category,Price,Stock,Emoji,Color
0,1,Orange Juice,Beverages,70,200,🍊,#ff6b6b
1,2,Apple Juice,Beverages,80,150,🍎,#ffd93d
2,3,Cold Coffee,Beverages,60,180,☕,#6bcb77
3,4,Mineral Water,Beverages,20,300,💧,#4d96ff
4,5,Apple,Fruits,30,250,🍎,#ff6b6b
5,6,Banana,Fruits,10,400,🍌,#ffd93d
6,7,Orange,Fruits,20,200,🍊,#ff6b6b
7,8,Grapes,Fruits,50,150,🍇,#9b59b6
8,9,Potato,Vegetables,20,300,🥔,#e67e22
9,10,Tomato,Vegetables,30,250,🍅,#e74c3c


In [ ]:
# CELL 2: SHOPPING MALL FUNCTIONS

def show_products():
    """Display all available products."""
    print("🛍️ AVAILABLE PRODUCTS")
    display(products_df[[
        "Product_ID", "Name", "Category", "Price", "Stock"
    ]])


def add_to_cart(product_id, quantity):
    """Add a product to the shopping cart."""
    global balance, products_df, cart

    try:
        product_id = int(product_id)
        quantity = int(quantity)
    except (ValueError, TypeError):
        print("❌ Enter a valid product and quantity.")
        return

    if quantity <= 0:
        print("❌ Quantity must be greater than zero.")
        return

    matching = products_df.index[
        products_df["Product_ID"] == product_id
    ].tolist()

    if not matching:
        print("❌ Product not found.")
        return

    idx = matching[0]
    product = products_df.loc[idx]

    if quantity > int(product["Stock"]):
        print(f"❌ Insufficient stock. Available: {product['Stock']}")
        return

    total_cost = quantity * float(product["Price"])

    if total_cost > balance:
        print(
            f"❌ Insufficient balance!\n"
            f"Required: ₹{total_cost:.2f}\n"
            f"Available: ₹{balance:.2f}"
        )
        return

    # Reduce available stock during this shopping session
    products_df.loc[idx, "Stock"] -= quantity
    balance -= total_cost

    if product_id in cart:
        cart[product_id]["Quantity"] += quantity
        cart[product_id]["Total"] += total_cost
    else:
        cart[product_id] = {
            "Name": product["Name"],
            "Emoji": product["Emoji"],
            "Quantity": quantity,
            "Price": float(product["Price"]),
            "Total": total_cost
        }

    print(f"✅ {product['Name']} added to your cart!")
    print(f"Remaining balance: ₹{balance:.2f}")


def show_cart():
    """Display the current cart."""
    if not cart:
        print("🛒 Your cart is empty.")
        return

    cart_data = []

    for item in cart.values():
        cart_data.append({
            "Product": item["Name"],
            "Quantity": item["Quantity"],
            "Unit Price": item["Price"],
            "Subtotal": item["Total"]
        })

    print("🛍️ YOUR SHOPPING CART")
    display(pd.DataFrame(cart_data))

    total = sum(item["Total"] for item in cart.values())
    print(f"Cart Total: ₹{total:.2f}")
    print(f"Remaining Balance: ₹{balance:.2f}")


def clear_cart():
    """Remove all items and restore stock and balance."""
    global balance, products_df, cart

    if not cart:
        print("⚠️ Your cart is already empty.")
        return

    for product_id, item in cart.items():
        idx = products_df.index[
            products_df["Product_ID"] == product_id
        ][0]

        products_df.loc[idx, "Stock"] += item["Quantity"]
        balance += item["Total"]

    cart.clear()

    print("✅ Cart cleared successfully!")
    print(f"Balance restored: ₹{balance:.2f}")


def checkout(customer_name, phone=""):
    """Complete purchase and save customer and sales records."""
    global balance, products_df, cart

    if not cart:
        print("⚠️ Add products to your cart before checkout.")
        return

    customer_name = str(customer_name).strip()
    phone = str(phone).strip()

    if not customer_name:
        print("❌ Please enter the customer's name.")
        return

    total_spent = sum(item["Total"] for item in cart.values())
    transaction_id = str(uuid.uuid4())[:8].upper()
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Prepare sales records
    sales_rows = []

    for item in cart.values():
        sales_rows.append({
            "Transaction_ID": transaction_id,
            "Date": current_time,
            "Customer_Name": customer_name,
            "Phone": phone,
            "Product_Name": item["Name"],
            "Quantity": item["Quantity"],
            "Unit_Price": item["Price"],
            "Total": item["Total"]
        })

    sales_df = pd.read_csv("sales.csv")
    sales_df = pd.concat(
        [sales_df, pd.DataFrame(sales_rows)],
        ignore_index=True
    )
    sales_df.to_csv("sales.csv", index=False)

    # Save updated inventory
    products_df.to_csv("products.csv", index=False)

    # Update customer database
    customers_df = pd.read_csv("customers.csv")

    if phone:
        customer_matches = customers_df.index[
            customers_df["Phone"].astype(str) == phone
        ].tolist()
    else:
        customer_matches = []

    if customer_matches:
        customer_idx = customer_matches[0]
        customers_df.loc[customer_idx, "Total_Spent"] = (
            float(customers_df.loc[customer_idx, "Total_Spent"])
            + total_spent
        )
        customers_df.loc[customer_idx, "Visits"] = (
            int(customers_df.loc[customer_idx, "Visits"]) + 1
        )
        customers_df.loc[customer_idx, "Last_Visit"] = current_time
        customers_df.loc[customer_idx, "Name"] = customer_name
    else:
        new_customer = pd.DataFrame([{
            "Customer_ID": str(uuid.uuid4())[:8].upper(),
            "Name": customer_name,
            "Phone": phone,
            "Total_Spent": total_spent,
            "Visits": 1,
            "Last_Visit": current_time
        }])

        customers_df = pd.concat(
            [customers_df, new_customer],
            ignore_index=True
        )

    customers_df.to_csv("customers.csv", index=False)

    # Generate receipt
    print("\n" + "=" * 45)
    print("           🛒 SHOPPING MALL RECEIPT")
    print("=" * 45)
    print(f"Transaction ID: {transaction_id}")
    print(f"Customer: {customer_name}")
    print(f"Date: {current_time}")
    print("-" * 45)

    for item in cart.values():
        print(f"{item['Emoji']} {item['Name']}")
        print(f"   Quantity: {item['Quantity']}")
        print(f"   Price: ₹{item['Price']:.2f}")
        print(f"   Total: ₹{item['Total']:.2f}")

    print("-" * 45)
    print(f"Total Spent: ₹{total_spent:.2f}")
    print(f"Remaining Balance: ₹{balance:.2f}")
    print("=" * 45)
    print("✨ Thank you for shopping! ✨")

    cart.clear()


def sales_report():
    """Display sales summary and best-selling product."""
    if not os.path.exists("sales.csv"):
        print("No sales data available.")
        return

    sales_df = pd.read_csv("sales.csv")

    if sales_df.empty:
        print("⚠️ No completed sales yet.")
        return

    revenue = sales_df["Total"].sum()
    quantity_sold = sales_df["Quantity"].sum()
    transactions = sales_df["Transaction_ID"].nunique()

    best_seller = (
        sales_df.groupby("Product_Name")["Quantity"]
        .sum()
        .idxmax()
    )

    print("📊 SALES REPORT")
    print("-" * 35)
    print(f"Total Revenue: ₹{revenue:.2f}")
    print(f"Total Items Sold: {quantity_sold}")
    print(f"Total Transactions: {transactions}")
    print(f"Best-Selling Product: {best_seller}")

    # Product-wise sales graph
    product_sales = sales_df.groupby("Product_Name")["Quantity"].sum()

    plt.figure(figsize=(11, 5))
    product_sales.sort_values(ascending=False).plot(kind="bar")
    plt.title("Shopping Mall - Product Sales")
    plt.xlabel("Products")
    plt.ylabel("Quantity Sold")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    print("\n📁 Customer Records:")
    display(pd.read_csv("customers.csv"))

    print("\n📁 Sales Records:")
    display(sales_df)

In [ ]:
# CELL 3: CREATE INTERACTIVE SHOPPING MALL INTERFACE

# Main heading
heading = widgets.HTML("""
<div style="
    background-color:#1a1a2e;
    color:white;
    padding:20px;
    text-align:center;
    border-radius:12px;">
    <h1>🛒 SHOPPING MALL</h1>
    <h3>Premium Shopping Experience</h3>
</div>
""")

# Balance display
balance_display = widgets.HTML()

def update_balance_display():
    balance_display.value = f"""
    <div style="
        background-color:#16213e;
        color:#00ff88;
        padding:15px;
        border-radius:10px;
        text-align:center;">
        <h3>💰 YOUR REMAINING BALANCE</h3>
        <h2>₹{balance:.2f}</h2>
    </div>
    """

update_balance_display()

# Category selector
category_dropdown = widgets.Dropdown(
    options=["All Categories"] + sorted(products_df["Category"].unique().tolist()),
    description="Category:",
    layout=widgets.Layout(width="350px")
)

# Product selector
product_dropdown = widgets.Dropdown(
    description="Product:",
    layout=widgets.Layout(width="350px")
)

# Quantity selector
quantity_input = widgets.BoundedIntText(
    value=1,
    min=1,
    max=1000,
    description="Quantity:",
    layout=widgets.Layout(width="250px")
)

# Customer details
customer_name_input = widgets.Text(
    placeholder="Enter customer name",
    description="Name:",
    layout=widgets.Layout(width="350px")
)

phone_input = widgets.Text(
    placeholder="Enter phone number (optional)",
    description="Phone:",
    layout=widgets.Layout(width="350px")
)

# Output areas
product_output = widgets.Output()
cart_output = widgets.Output()
message_output = widgets.Output()
report_output = widgets.Output()

def update_product_options(change=None):
    """Update product list according to selected category."""
    category = category_dropdown.value

    if category == "All Categories":
        filtered = products_df
    else:
        filtered = products_df[
            products_df["Category"] == category
        ]

    options = []

    for _, row in filtered.iterrows():
        label = (
            f"{row['Emoji']} {row['Name']} "
            f"- ₹{row['Price']} | Stock: {row['Stock']}"
        )
        options.append((label, int(row["Product_ID"])))

    product_dropdown.options = options

    if options:
        quantity_input.value = 1
        quantity_input.disabled = False
    else:
        quantity_input.disabled = True


def refresh_product_table():
    """Refresh the product table."""
    with product_output:
        clear_output(wait=True)

        print("✨ AVAILABLE PRODUCTS")

        if category_dropdown.value == "All Categories":
            filtered = products_df
        else:
            filtered = products_df[
                products_df["Category"] == category_dropdown.value
            ]

        display(filtered[[
            "Product_ID", "Name", "Category", "Price", "Stock"
        ]].reset_index(drop=True))


def refresh_cart():
    """Refresh the cart display."""
    with cart_output:
        clear_output(wait=True)
        show_cart()


def add_button_clicked(button):
    with message_output:
        clear_output(wait=True)

        if product_dropdown.value is None:
            print("❌ No product available in this category.")
            return

        add_to_cart(
            product_dropdown.value,
            quantity_input.value
        )

    update_balance_display()
    update_product_options()
    refresh_product_table()
    refresh_cart()


def clear_button_clicked(button):
    with message_output:
        clear_output(wait=True)
        clear_cart()

    update_balance_display()
    update_product_options()
    refresh_product_table()
    refresh_cart()


def checkout_button_clicked(button):
    with message_output:
        clear_output(wait=True)

        checkout(
            customer_name_input.value,
            phone_input.value
        )

    update_balance_display()
    update_product_options()
    refresh_product_table()
    refresh_cart()


def report_button_clicked(button):
    with report_output:
        clear_output(wait=True)
        sales_report()


# Buttons
add_button = widgets.Button(
    description="🛒 ADD TO CART",
    button_style="success",
    layout=widgets.Layout(width="200px", height="45px")
)

clear_button = widgets.Button(
    description="🗑️ CLEAR CART",
    button_style="danger",
    layout=widgets.Layout(width="200px", height="45px")
)

checkout_button = widgets.Button(
    description="✨ CHECKOUT",
    button_style="success",
    layout=widgets.Layout(width="200px", height="45px")
)

report_button = widgets.Button(
    description="📊 SALES REPORT",
    button_style="info",
    layout=widgets.Layout(width="200px", height="45px")
)

# Connect buttons to functions
add_button.on_click(add_button_clicked)
clear_button.on_click(clear_button_clicked)
checkout_button.on_click(checkout_button_clicked)
report_button.on_click(report_button_clicked)

# Update products when category changes
category_dropdown.observe(
    update_product_options,
    names="value"
)

# Initial display
update_product_options()
refresh_product_table()

# Layout
display(heading)
display(balance_display)

display(widgets.HTML("<h2>🛍️ Select Your Products</h2>"))
display(category_dropdown)
display(product_dropdown)
display(quantity_input)
display(add_button)

display(product_output)

display(widgets.HTML("<h2>🛒 Shopping Cart</h2>"))
display(cart_output)
display(clear_button)

display(widgets.HTML("<h2>👤 Customer Details</h2>"))
display(customer_name_input)
display(phone_input)
display(checkout_button)

display(message_output)

display(widgets.HTML("<h2>📈 Business Analytics</h2>"))
display(report_button)
display(report_output)

HTML(value='\n<div style="\n    background-color:#1a1a2e;\n    color:white;\n    padding:20px;\n    text-align…

HTML(value='\n    <div style="\n        background-color:#16213e;\n        color:#00ff88;\n        padding:15p…

HTML(value='<h2>🛍️ Select Your Products</h2>')

Dropdown(description='Category:', layout=Layout(width='350px'), options=('All Categories', 'Beverages', 'Dairy…

Dropdown(description='Product:', layout=Layout(width='350px'), options=(('🍊 Orange Juice - ₹70 | Stock: 200', …

BoundedIntText(value=1, description='Quantity:', layout=Layout(width='250px'), max=1000, min=1)

... (9 outputs left out)

Output()

HTML(value='<h2>📈 Business Analytics</h2>')

Button(button_style='info', description='📊 SALES REPORT', layout=Layout(height='45px', width='200px'), style=B…

Output()

In [ ]:
from google.colab import files

files.download("products.csv")
files.download("customers.csv")
files.download("sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>